# UPF Literature — Author Co-authorship Network

Builds and analyses a co-authorship network from the edge list produced by
`upf_bibliometrics.py`.  
**Pre-requisite:** run `python upf_bibliometrics.py` (or `--dry-run` for a
quick test) so that `output/coauthorship_edges.csv` exists.

**Install once** (if not already present):
```bash
pip install networkx plotly
```

**To export the interactive HTML dashboard**, run all cells then open
`output/upf_dashboard.html` in any browser — no Python required.
(`nbconvert --to html` renders the notebook with code cells; the dashboard
cell at the bottom produces the clean standalone file instead.)

In [ ]:
import collections
import math
import os
import warnings

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import networkx as nx
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Use CDN-backed renderer so nbconvert can embed charts as HTML
pio.renderers.default = 'notebook_connected'

warnings.filterwarnings('ignore')

# ── Paths (relative to notebook location) ─────────────────────────────────────
EDGES_CSV        = os.path.join('..', 'output', 'coauthorship_edges.csv')
AUTHORS_CSV      = os.path.join('..', 'output', 'papers_by_author.csv')
INSTITUTIONS_CSV    = os.path.join('..', 'output', 'papers_by_institution.csv')
FUNDERS_CSV         = os.path.join('..', 'output', 'papers_by_funder.csv')
FUNDING_COUNTRY_CSV = os.path.join('..', 'output', 'funding_by_country.csv')
OUTPUT_DIR          = os.path.join('..', 'output')

# ── Tunable parameters ─────────────────────────────────────────────────────────
MIN_PAPERS      = 3    # keep only authors with >= this many papers in the corpus
MIN_EDGE_WEIGHT = 2    # keep only co-authorship pairs that share >= this many papers
TOP_N_LABELS    = 30   # label the top-N highest-degree nodes in the graph plot
TOP_N_RANKING   = 25   # entries shown in the institution / author ranking charts
LAYOUT_SEED     = 42

print('NetworkX version:', nx.__version__)
print('Plotly version  :', plotly.__version__)

## 0. Quick Search

Set `SEARCH_AUTHOR` or `SEARCH_INSTITUTION` to a substring (case-insensitive)
and run this cell to look up any name before diving into the full analysis.

In [ ]:
SEARCH_AUTHOR      = ""   # e.g. "Monteiro"  or  "Touvier"
SEARCH_INSTITUTION = ""   # e.g. "São Paulo"  or  "Deakin"

def _search(df, col, query, extra_cols):
    if not query.strip():
        return None
    mask = df[col].str.contains(query, case=False, na=False)
    result = df[mask].sort_values('papers', ascending=False)
    return result[extra_cols]

# Authors
if SEARCH_AUTHOR:
    res = _search(
        authors_df, 'author_name', SEARCH_AUTHOR,
        ['author_name', 'institution', 'country', 'papers', 'citations']
    )
    print(f"Authors matching '{SEARCH_AUTHOR}':")
    display(res.head(20)) if res is not None and len(res) else print("  (no matches)")

# Institutions
if SEARCH_INSTITUTION:
    res = _search(
        institutions_df, 'institution', SEARCH_INSTITUTION,
        ['institution', 'country', 'papers', 'citations']
    )
    print(f"Institutions matching '{SEARCH_INSTITUTION}':")
    display(res.head(20)) if res is not None and len(res) else print("  (no matches)")

if not SEARCH_AUTHOR and not SEARCH_INSTITUTION:
    print("Set SEARCH_AUTHOR or SEARCH_INSTITUTION above and re-run.")

## 1. Load data

In [ ]:
edges_df        = pd.read_csv(EDGES_CSV)
authors_df      = pd.read_csv(AUTHORS_CSV)
institutions_df = pd.read_csv(INSTITUTIONS_CSV)
try:
    funders_df     = pd.read_csv(FUNDERS_CSV)
    funding_cty_df = pd.read_csv(FUNDING_COUNTRY_CSV)
except FileNotFoundError:
    funders_df = funding_cty_df = pd.DataFrame()
    print('⚠ Funding CSVs not found — re-run upf_bibliometrics.py first')

# Drop the "no institution" catch-all row
institutions_df = institutions_df[
    institutions_df['institution'].notna() & (institutions_df['institution'] != '')
].copy()

print(f'Edge rows         : {len(edges_df):,}')
print(f'Author rows       : {len(authors_df):,}')
print(f'Institution rows  : {len(institutions_df):,}')
edges_df.head(3)

## 2. Build the full graph, then filter

We keep only nodes (authors) that appear in at least `MIN_PAPERS` papers **and**
edges (co-authorship pairs) with at least `MIN_EDGE_WEIGHT` shared papers.
This removes noise from one-off collaborations and focuses the network on
the productive core of the field.

In [ ]:
# Node attribute lookup: author_id → {name, institution, country, papers}
# groupby+first handles any duplicate author_id rows (e.g. authors with no
# OpenAlex ID whose fallback key collides across works)
node_attrs = (
    authors_df
    .groupby('author_id', as_index=True)
    .first()[['author_name', 'institution', 'country', 'papers', 'citations']]
    .to_dict(orient='index')
)

# Productive-author set
core_authors = {
    aid for aid, attr in node_attrs.items()
    if attr['papers'] >= MIN_PAPERS
}
print(f'Authors with >= {MIN_PAPERS} papers : {len(core_authors):,}')

# Build graph
G = nx.Graph()

for _, row in edges_df.iterrows():
    a1, a2, w = row['author1_id'], row['author2_id'], row['shared_papers']
    if a1 not in core_authors or a2 not in core_authors:
        continue
    if w < MIN_EDGE_WEIGHT:
        continue
    if G.has_edge(a1, a2):
        G[a1][a2]['weight'] += w
    else:
        G.add_edge(a1, a2, weight=w)

# Attach node attributes
for node in G.nodes():
    attrs = node_attrs.get(node, {})
    G.nodes[node]['name']        = attrs.get('author_name', node)
    G.nodes[node]['institution'] = attrs.get('institution', '')
    G.nodes[node]['country']     = attrs.get('country', '')
    G.nodes[node]['papers']      = attrs.get('papers', 0)
    G.nodes[node]['citations']    = attrs.get('citations', 0)

# Focus on the largest connected component
lcc_nodes = max(nx.connected_components(G), key=len)
G_lcc     = G.subgraph(lcc_nodes).copy()

print(f'Full graph        : {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
print(f'Largest component : {G_lcc.number_of_nodes():,} nodes, {G_lcc.number_of_edges():,} edges')


## 3. Global network statistics

In [4]:
n = G_lcc.number_of_nodes()
m = G_lcc.number_of_edges()
density   = nx.density(G_lcc)
avg_deg   = 2 * m / n if n else 0
avg_clust = nx.average_clustering(G_lcc, weight='weight')
diameter  = nx.diameter(G_lcc) if n < 5_000 else 'skipped (graph too large)'
avg_path  = nx.average_shortest_path_length(G_lcc) if n < 5_000 else 'skipped'
components_full = nx.number_connected_components(G)

print('── Network statistics (largest component) ──────────────')
print(f'  Nodes                 : {n:,}')
print(f'  Edges                 : {m:,}')
print(f'  Density               : {density:.4f}')
print(f'  Average degree        : {avg_deg:.2f}')
print(f'  Average clustering    : {avg_clust:.4f}')
print(f'  Diameter              : {diameter}')
print(f'  Avg shortest path     : {avg_path}')
print(f'  Components (full graph): {components_full:,}')

── Network statistics (largest component) ──────────────
  Nodes                 : 1,380
  Edges                 : 8,921
  Density               : 0.0094
  Average degree        : 12.93
  Average clustering    : 0.0397
  Diameter              : 13
  Avg shortest path     : 4.446804552763502
  Components (full graph): 172


## 4. Centrality measures

In [ ]:
degree_cent     = nx.degree_centrality(G_lcc)
betweenness     = nx.betweenness_centrality(G_lcc, weight='weight', normalized=True)
pagerank        = nx.pagerank(G_lcc, weight='weight')
clustering      = nx.clustering(G_lcc, weight='weight')

centrality_df = pd.DataFrame({
    'author_id'   : list(G_lcc.nodes()),
    'name'        : [G_lcc.nodes[n]['name']        for n in G_lcc.nodes()],
    'institution' : [G_lcc.nodes[n]['institution'] for n in G_lcc.nodes()],
    'country'     : [G_lcc.nodes[n]['country']     for n in G_lcc.nodes()],
    'papers'      : [G_lcc.nodes[n]['papers']      for n in G_lcc.nodes()],
    'citations'   : [G_lcc.nodes[n].get('citations', 0) for n in G_lcc.nodes()],
    'degree'      : [G_lcc.degree(n)               for n in G_lcc.nodes()],
    'degree_centrality'  : [degree_cent[n]   for n in G_lcc.nodes()],
    'betweenness'        : [betweenness[n]   for n in G_lcc.nodes()],
    'pagerank'           : [pagerank[n]      for n in G_lcc.nodes()],
    'clustering'         : [clustering[n]   for n in G_lcc.nodes()],
})

centrality_df.sort_values('betweenness', ascending=False, inplace=True)
centrality_df.reset_index(drop=True, inplace=True)

# Save
out_path = os.path.join(OUTPUT_DIR, 'author_centrality.csv')
centrality_df.to_csv(out_path, index=False)
print(f'Saved → {out_path}')

print('\nTop 15 by betweenness centrality (bridges / gatekeepers):')
centrality_df[['name', 'institution', 'country', 'papers', 'degree', 'betweenness', 'pagerank']].head(15)

In [6]:
print('Top 15 by degree (most direct collaborators):')
centrality_df.sort_values('degree', ascending=False).head(15)[
    ['name', 'institution', 'country', 'papers', 'degree', 'betweenness']
]

Top 15 by degree (most direct collaborators):


,name,institution,country,papers,degree,betweenness
2,Renata Bertazzi Levy,Universidade de São Paulo,BR,143,165,0.107005
1,Carlos Augusto Monteiro,Universidade de São Paulo,BR,156,139,0.129257
4,Fernanda Rauber,Universidade de São Paulo,BR,93,138,0.098458
5,Eurídice Martínez Steele,Universidade de São Paulo,BR,133,120,0.096528
0,Neha Khandpur,Universidade de São Paulo,BR,96,116,0.142175
16,Christopher Millett,Universidade de São Paulo,BR,45,115,0.043840
40,Mathilde Touvier,Sorbonne Paris Cité,FR,71,105,0.022435
29,Bernard Srour,Equipe de Recherche en Epidémiologie Nutrition...,FR,61,103,0.029632
50,Inge Huybrechts,Centre international de recherche sur le cancer,FR,33,86,0.018655
91,Eszter P. Vamos,Imperial College London,GB,35,81,0.008897


## 5. Top-N Rankings — Institutions and Authors

Interactive bar charts. Hover over any bar for full details.  
Change `TOP_N_RANKING` in the parameters cell (cell 1) to show more or fewer entries.

In [7]:
# ── Top-N Institutions ────────────────────────────────────────────────────────
inst_by_papers = (
    institutions_df.nlargest(TOP_N_RANKING, 'papers')
    .assign(label=lambda df: df['institution'].str[:55])
    .sort_values('papers')          # ascending → largest at top in horizontal bar
)
inst_by_cites = (
    institutions_df.nlargest(TOP_N_RANKING, 'citations')
    .assign(label=lambda df: df['institution'].str[:55])
    .sort_values('citations')
)

fig_inst = go.Figure()

fig_inst.add_trace(go.Bar(
    x=inst_by_papers['papers'],
    y=inst_by_papers['label'],
    orientation='h',
    name='Papers',
    visible=True,
    customdata=inst_by_papers[['institution', 'country', 'citations']].values,
    hovertemplate=(
        '<b>%{customdata[0]}</b><br>'
        'Country: %{customdata[1]}<br>'
        'Papers: %{x:,}<br>'
        'Citations: %{customdata[2]:,}<extra></extra>'
    ),
    marker_color='steelblue',
))

fig_inst.add_trace(go.Bar(
    x=inst_by_cites['citations'],
    y=inst_by_cites['label'],
    orientation='h',
    name='Citations',
    visible=False,
    customdata=inst_by_cites[['institution', 'country', 'papers']].values,
    hovertemplate=(
        '<b>%{customdata[0]}</b><br>'
        'Country: %{customdata[1]}<br>'
        'Citations: %{x:,}<br>'
        'Papers: %{customdata[2]:,}<extra></extra>'
    ),
    marker_color='coral',
))

fig_inst.update_layout(
    title=f'Top {TOP_N_RANKING} Institutions by Paper Count',
    height=700,
    xaxis_title='Papers',
    yaxis_title='',
    showlegend=False,
    margin=dict(l=10, r=20, t=60, b=40),
    updatemenus=[dict(
        type='buttons',
        direction='right',
        x=0.0, xanchor='left',
        y=1.08, yanchor='top',
        buttons=[
            dict(
                label='By Papers',
                method='update',
                args=[{'visible': [True, False]},
                      {'title': f'Top {TOP_N_RANKING} Institutions by Paper Count',
                       'xaxis.title.text': 'Papers'}],
            ),
            dict(
                label='By Citations',
                method='update',
                args=[{'visible': [False, True]},
                      {'title': f'Top {TOP_N_RANKING} Institutions by Citation Count',
                       'xaxis.title.text': 'Citations'}],
            ),
        ],
    )],
)
fig_inst.show()

In [8]:
# ── Top-N Authors ─────────────────────────────────────────────────────────────
authors_clean = authors_df[
    authors_df['author_name'].notna() & (authors_df['author_name'] != '')
].copy()

auth_by_papers = (
    authors_clean.nlargest(TOP_N_RANKING, 'papers')
    .sort_values('papers')
)
auth_by_cites = (
    authors_clean.nlargest(TOP_N_RANKING, 'citations')
    .sort_values('citations')
)

fig_auth = go.Figure()

fig_auth.add_trace(go.Bar(
    x=auth_by_papers['papers'],
    y=auth_by_papers['author_name'],
    orientation='h',
    name='Papers',
    visible=True,
    customdata=auth_by_papers[['institution', 'country', 'citations']].values,
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Institution: %{customdata[0]}<br>'
        'Country: %{customdata[1]}<br>'
        'Papers: %{x:,}<br>'
        'Citations: %{customdata[2]:,}<extra></extra>'
    ),
    marker_color='steelblue',
))

fig_auth.add_trace(go.Bar(
    x=auth_by_cites['citations'],
    y=auth_by_cites['author_name'],
    orientation='h',
    name='Citations',
    visible=False,
    customdata=auth_by_cites[['institution', 'country', 'papers']].values,
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Institution: %{customdata[0]}<br>'
        'Country: %{customdata[1]}<br>'
        'Citations: %{x:,}<br>'
        'Papers: %{customdata[2]:,}<extra></extra>'
    ),
    marker_color='coral',
))

fig_auth.update_layout(
    title=f'Top {TOP_N_RANKING} Authors by Paper Count',
    height=700,
    xaxis_title='Papers',
    yaxis_title='',
    showlegend=False,
    margin=dict(l=10, r=20, t=60, b=40),
    updatemenus=[dict(
        type='buttons',
        direction='right',
        x=0.0, xanchor='left',
        y=1.08, yanchor='top',
        buttons=[
            dict(
                label='By Papers',
                method='update',
                args=[{'visible': [True, False]},
                      {'title': f'Top {TOP_N_RANKING} Authors by Paper Count',
                       'xaxis.title.text': 'Papers'}],
            ),
            dict(
                label='By Citations',
                method='update',
                args=[{'visible': [False, True]},
                      {'title': f'Top {TOP_N_RANKING} Authors by Citation Count',
                       'xaxis.title.text': 'Citations'}],
            ),
        ],
    )],
)
fig_auth.show()

## 5. Community detection

Uses the Louvain algorithm (built into NetworkX ≥ 2.7) to partition the
network into research communities.

In [ ]:
communities = nx.community.louvain_communities(G_lcc, weight='weight', seed=LAYOUT_SEED)
communities = sorted(communities, key=len, reverse=True)

print(f'Number of communities detected: {len(communities)}')
print(f'Sizes of top-10 communities   : {[len(c) for c in communities[:10]]}')

# Assign community labels to nodes
node_community = {}
for idx, community in enumerate(communities):
    for node in community:
        node_community[node] = idx

nx.set_node_attributes(G_lcc, node_community, 'community')
centrality_df['community'] = centrality_df['author_id'].map(node_community)

# Build per-community summary
comm_summary = (
    centrality_df
    .groupby('community')
    .agg(
        size=('name', 'count'),
        total_papers=('papers', 'sum'),
        total_citations=('citations', 'sum'),
        top_country=('country', lambda x: x.value_counts().index[0] if len(x) else ''),
        top_institution=('institution', lambda x: x.value_counts().index[0] if len(x) else ''),
        # Top author by betweenness within the community
        top_author=('name', lambda x: (
            centrality_df.loc[x.index]
            .sort_values('betweenness', ascending=False)['name'].iloc[0]
            if len(x) else ''
        )),
    )
    .reset_index()
    .sort_values('size', ascending=False)
)

# Build human-readable community labels: "Country · Lastname (N)"
def _make_label(row):
    lastname = row['top_author'].split()[-1] if row['top_author'] else '?'
    country  = row['top_country'] or '?'
    n        = int(row['size'])
    return f"{country} · {lastname} ({n})"

community_names = {
    int(row['community']): _make_label(row)
    for _, row in comm_summary.iterrows()
}
nx.set_node_attributes(G_lcc, {n: community_names.get(v, str(v))
                                for n, v in node_community.items()}, 'community_label')
centrality_df['community_label'] = centrality_df['community'].map(community_names)

print('\nTop-10 communities:')
comm_summary[['community', 'size', 'total_papers', 'total_citations',
              'top_country', 'top_author']].head(10)

## 6. Visualisations

### 6a. Degree distribution

In [10]:
degrees = [d for _, d in G_lcc.degree()]
freq    = collections.Counter(degrees)
deg_df  = pd.DataFrame({'degree': list(freq.keys()), 'count': list(freq.values())}).sort_values('degree')

fig_deg = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Degree Distribution (linear)', 'Degree Distribution (log–log)'],
)

fig_deg.add_trace(
    go.Histogram(x=degrees, nbinsx=40, name='Count', marker_color='steelblue'),
    row=1, col=1,
)
fig_deg.add_trace(
    go.Scatter(
        x=deg_df['degree'], y=deg_df['count'],
        mode='markers',
        marker=dict(size=5, color='steelblue', opacity=0.7),
        hovertemplate='Degree: %{x}<br>Count: %{y}<extra></extra>',
        name='Count',
    ),
    row=1, col=2,
)

fig_deg.update_xaxes(title_text='Degree', row=1, col=1)
fig_deg.update_yaxes(title_text='Count', row=1, col=1)
fig_deg.update_xaxes(type='log', title_text='Degree (log)', row=1, col=2)
fig_deg.update_yaxes(type='log', title_text='Count (log)', row=1, col=2)
fig_deg.update_layout(
    height=420,
    showlegend=False,
    title_text=(
        f'Degree Distribution  |  '
        f'Mean: {sum(degrees)/len(degrees):.2f}  |  Max: {max(degrees)}'
    ),
)
fig_deg.show()
print(f'Mean degree: {sum(degrees)/len(degrees):.2f}  Max: {max(degrees)}')

Mean degree: 12.93  Max: 165


**Reading this chart:**  
The left panel shows how many authors have each number of co-authors (degree).
A long right tail — with a few authors having very many collaborators and most
having only a few — is typical of real-world collaboration networks and suggests
a **scale-free** structure.  
The log–log panel (right) linearises that tail: if the points follow a straight
line it is consistent with a power-law degree distribution, which characterises
highly unequal networks where a small hub dominates.  
**What to look for:** a very steep slope means collaboration is concentrated in
a few prolific connectors; a shallower slope indicates a more distributed network.

### 6b. Co-authorship network (largest component, coloured by community)

In [ ]:
# Cap for legibility
PLOT_CAP = 500
if G_lcc.number_of_nodes() > PLOT_CAP:
    top_nodes = sorted(G_lcc.nodes(), key=lambda n: G_lcc.degree(n), reverse=True)[:PLOT_CAP]
    G_plot = G_lcc.subgraph(top_nodes).copy()
    print(f'Plotting top-{PLOT_CAP} nodes by degree ({G_lcc.number_of_nodes():,} total)')
else:
    G_plot = G_lcc

pos = nx.spring_layout(G_plot, weight='weight', seed=LAYOUT_SEED, k=0.8)

# ── Build Plotly traces ───────────────────────────────────────────────────────
# Edges
edge_x, edge_y = [], []
for u, v in G_plot.edges():
    x0, y0 = pos[u]; x1, y1 = pos[v]
    edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

edge_trace = go.Scatter(
    x=edge_x, y=edge_y, mode='lines',
    line=dict(width=0.5, color='#888'), hoverinfo='none',
    showlegend=False,
)

# Nodes — one trace per community for legend
comm_ids = sorted(set(G_plot.nodes[n].get('community', 0) for n in G_plot.nodes()))
palette  = px.colors.qualitative.Alphabet
node_traces = []

for cid in comm_ids:
    nodes_in_comm = [n for n in G_plot.nodes() if G_plot.nodes[n].get('community', 0) == cid]
    label = G_plot.nodes[nodes_in_comm[0]].get('community_label', str(cid)) if nodes_in_comm else str(cid)
    xs = [pos[n][0] for n in nodes_in_comm]
    ys = [pos[n][1] for n in nodes_in_comm]
    sizes  = [6 + 2 * G_plot.degree(n) for n in nodes_in_comm]
    texts  = [G_plot.nodes[n].get('name', n).split()[-1]
               if G_plot.degree(n) >= sorted([G_plot.degree(n) for n in G_plot.nodes()])[-TOP_N_LABELS]
               else '' for n in nodes_in_comm]
    hovers = [f"{G_plot.nodes[n].get('name','')}<br>{G_plot.nodes[n].get('institution','')}<br>"
               f"{G_plot.nodes[n].get('country','')}  deg={G_plot.degree(n)}"
               for n in nodes_in_comm]
    node_traces.append(go.Scatter(
        x=xs, y=ys, mode='markers+text',
        marker=dict(size=sizes, color=palette[cid % len(palette)], opacity=0.85,
                    line=dict(width=0.5, color='white')),
        text=texts, textposition='top center', textfont=dict(size=7),
        hovertext=hovers, hoverinfo='text',
        name=label, legendgroup=str(cid),
    ))

fig_network = go.Figure(data=[edge_trace] + node_traces)
fig_network.update_layout(
    title=f'UPF Co-authorship Network  |  {G_plot.number_of_nodes()} authors, '
          f'{G_plot.number_of_edges()} edges  |  colour = community',
    showlegend=True,
    hovermode='closest',
    height=750,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    legend=dict(title='Community', itemsizing='constant', font=dict(size=9)),
    margin=dict(l=10, r=10, t=50, b=10),
)
fig_network.show()

**Reading this chart:**  
Each dot is an author; edges connect co-authors.  
**Colour** groups authors into research communities detected by the Louvain
algorithm — nodes of the same colour tend to publish together more than with
other groups.  
**Size** reflects degree (number of direct co-authors): large nodes are the most
collaborative hubs.  
**Layout** uses a force-directed spring algorithm: nodes that share many
collaborators are pulled close together, so spatial proximity approximates
research community.  
**What to look for:** tight clusters indicate nationally or institutionally
cohesive groups; bridges between clusters are authors who connect otherwise
separate research communities.

### 6c. Betweenness vs degree scatter — identifying bridges

In [ ]:
# community_label is already built in the community-detection cell
plot_df = centrality_df.copy()
# Fallback for any unmapped nodes
plot_df['community_label'] = plot_df['community_label'].fillna('Other')

fig_scatter = px.scatter(
    plot_df,
    x='degree',
    y='betweenness',
    size='papers',
    color='community_label',
    hover_name='name',
    hover_data={
        'institution': True,
        'country': True,
        'papers': True,
        'degree': True,
        'betweenness': ':.4f',
        'pagerank': ':.4f',
        'community_label': False,
    },
    title='Degree vs Betweenness Centrality  |  point size ∝ papers  |  colour = community',
    labels={
        'degree': 'Degree (number of co-authors)',
        'betweenness': 'Betweenness centrality',
        'community_label': 'Community',
    },
    color_discrete_sequence=px.colors.qualitative.Alphabet,
    size_max=30,
    height=650,
)
fig_scatter.update_traces(marker_opacity=0.7)
fig_scatter.update_layout(legend_title_text='Community')
fig_scatter.show()

**Reading this chart:**  
- **X axis (degree):** how many distinct co-authors an author has.  
  High degree = a well-connected hub who works across many collaborations.  
- **Y axis (betweenness centrality):** how often an author lies on the shortest
  path between two other authors.  
  High betweenness = a *broker* or *bridge* — someone who connects otherwise
  separate research groups, even if they do not have the most co-authors overall.  
- **Point size** is proportional to the number of papers published.  
- **Colour** identifies the Louvain community (label = country · top author).  

**Four quadrant interpretation:**  
| | Low betweenness | High betweenness |
|---|---|---|
| **High degree** | Core hub within one community | Cross-community connector (key bridge) |
| **Low degree** | Peripheral author | Structural hole broker |

Authors in the top-right are the most strategically important for knowledge
transfer across the field.

### 6d. Country-level collaboration heatmap

Counts co-authored papers between pairs of countries.

In [ ]:
def _safe_country(val):
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return ''
    return str(val)

country_edges = collections.Counter()
for u, v, data in G_lcc.edges(data=True):
    c1 = _safe_country(G_lcc.nodes[u].get('country', ''))
    c2 = _safe_country(G_lcc.nodes[v].get('country', ''))
    if c1 and c2 and c1 != c2:
        pair = tuple(sorted([c1, c2]))
        country_edges[pair] += data.get('weight', 1)

top_pairs = country_edges.most_common(20)
countries_involved = sorted({c for pair, _ in top_pairs for c in pair})

matrix = pd.DataFrame(0, index=countries_involved, columns=countries_involved)
for (c1, c2), w in top_pairs:
    matrix.loc[c1, c2] = w
    matrix.loc[c2, c1] = w

fig_heatmap = go.Figure(go.Heatmap(
    z=matrix.values.tolist(),
    x=list(matrix.columns),
    y=list(matrix.index),
    colorscale='YlOrRd',
    hoverongaps=False,
    hovertemplate='%{y} ↔ %{x}<br>Co-authored papers: %{z}<extra></extra>',
    colorbar=dict(title='Papers'),
))
fig_heatmap.update_layout(
    title='Cross-country Co-authorship (top-20 pairs)',
    height=520,
    xaxis=dict(tickangle=45),
    margin=dict(l=60, r=20, t=60, b=80),
)
fig_heatmap.show()

## 8. Citation Impact Analysis

The co-authorship network above captures *collaboration* structure.
This section uses the `citations` column to explore *impact* structure:
which authors, communities and countries generate the most-cited work,
and how efficiently (citations per paper).

> **Note on a full citation network:** A directed paper→paper citation graph
> would require fetching `referenced_works` for each paper from OpenAlex —
> an additional API pass of comparable scale to the main fetch.
> The analysis below uses the aggregate citation counts already retrieved.

In [ ]:
# ── 8a. Papers vs Citations scatter (author level) ────────────────────────────
plot_cit = centrality_df[centrality_df['papers'] >= MIN_PAPERS].copy()
plot_cit['cit_per_paper'] = (plot_cit['citations'] / plot_cit['papers']).round(1)
plot_cit['community_label'] = plot_cit['community_label'].fillna('Other')

fig_cit = px.scatter(
    plot_cit,
    x='papers',
    y='citations',
    size='degree',
    color='community_label',
    hover_name='name',
    hover_data={
        'institution': True,
        'country': True,
        'cit_per_paper': True,
        'degree': True,
        'community_label': False,
    },
    log_x=True,
    log_y=True,
    title='Author Citation Impact  |  size ∝ degree  |  colour = community<br>'
          '<sup>Log scales — top-right = many papers AND highly cited</sup>',
    labels={
        'papers': 'Papers (log)',
        'citations': 'Total citations (log)',
        'community_label': 'Community',
    },
    color_discrete_sequence=px.colors.qualitative.Alphabet,
    size_max=25,
    height=600,
)
fig_cit.update_traces(marker_opacity=0.7)
fig_cit.show()

# ── 8b. Citation efficiency by community ──────────────────────────────────────
comm_cit = (
    centrality_df
    .groupby('community_label')
    .agg(
        authors=('name', 'count'),
        total_papers=('papers', 'sum'),
        total_citations=('citations', 'sum'),
    )
    .assign(cit_per_paper=lambda df: (df['total_citations'] / df['total_papers']).round(1))
    .sort_values('total_citations', ascending=False)
    .head(15)
    .reset_index()
)

fig_comm_cit = go.Figure()
fig_comm_cit.add_trace(go.Bar(
    x=comm_cit['total_citations'],
    y=comm_cit['community_label'],
    orientation='h',
    name='Total citations',
    marker_color='steelblue',
    customdata=comm_cit[['total_papers', 'cit_per_paper', 'authors']].values,
    hovertemplate='<b>%{y}</b><br>Citations: %{x:,}<br>Papers: %{customdata[0]:,}'
                  '<br>Cit/paper: %{customdata[1]}<br>Authors: %{customdata[2]}<extra></extra>',
))
fig_comm_cit.update_layout(
    title='Total Citations by Research Community (top 15)',
    xaxis_title='Total citations',
    height=500,
    yaxis=dict(autorange='reversed'),
)
fig_comm_cit.show()

# ── 8c. Citations per paper by country ────────────────────────────────────────
country_df = pd.read_csv(os.path.join(OUTPUT_DIR, '..', 'output', 'papers_by_country.csv'))
country_df = country_df[country_df['country'].notna() & (country_df['country'] != '')].copy()
country_df['cit_per_paper'] = (country_df['citations'] / country_df['papers']).round(1)
top_countries = country_df.nlargest(20, 'papers')

fig_cty = make_subplots(rows=1, cols=2,
    subplot_titles=['Papers by Country (top 20)', 'Citations per Paper (top 20 by papers)'])

fig_cty.add_trace(go.Bar(
    x=top_countries['papers'], y=top_countries['country'],
    orientation='h', name='Papers', marker_color='steelblue',
    hovertemplate='<b>%{y}</b>: %{x:,} papers<extra></extra>',
), row=1, col=1)

fig_cty.add_trace(go.Bar(
    x=top_countries.sort_values('cit_per_paper', ascending=True)['cit_per_paper'],
    y=top_countries.sort_values('cit_per_paper', ascending=True)['country'],
    orientation='h', name='Cit/paper', marker_color='coral',
    hovertemplate='<b>%{y}</b>: %{x} cit/paper<extra></extra>',
), row=1, col=2)

fig_cty.update_layout(height=550, showlegend=False,
    title_text='Country-level Output vs Impact')
fig_cty.update_yaxes(autorange='reversed', row=1, col=1)
fig_cty.show()
print('\n**Interpretation:** Volume (papers) and impact (citations per paper) often diverge.\n'
      'Countries with fewer but more-cited papers punch above their weight in influence.')

**Reading these charts:**

**8a — Author citation impact (scatter):**  
Both axes are log-scaled. Authors in the **top-right** are highly prolific *and*
highly cited — the field's most influential researchers.  
Authors in the **top-left** have few papers but each is highly cited (early-career
or review-paper specialists); those in the **bottom-right** are prolific but
lower-impact.

**8b — Citations by community:**  
Shows which research cluster generates the most total citation impact.  
Compare `total_citations` with `total_papers` to see whether a community's
influence is driven by volume or quality.

**8c — Country output vs efficiency:**  
The left panel ranks by paper count (volume); the right panel shows citations
per paper (efficiency/quality).  
High volume + low efficiency may indicate a large but less internationally
visible community; low volume + high efficiency suggests a small but punchy
research base.

## 9. Funding Analysis

Which funders and countries drive the UPF literature?  
OpenAlex records the `grants` field where available — coverage is partial
(not all publishers deposit funding metadata), so interpret percentages as
lower bounds.

In [ ]:
# ── 9a. Top funders by paper count ───────────────────────────────────────────
top_funders = funders_df[funders_df['funder_name'].notna()].head(25).copy()
top_funders['cit_per_paper'] = (top_funders['citations'] / top_funders['papers']).round(1)
top_funders_plot = top_funders.sort_values('papers')

fig_funders = go.Figure()
fig_funders.add_trace(go.Bar(
    x=top_funders_plot['papers'],
    y=top_funders_plot['funder_name'],
    orientation='h',
    name='Papers',
    marker_color='steelblue',
    customdata=top_funders_plot[['citations', 'cit_per_paper']].values,
    hovertemplate='<b>%{y}</b><br>Papers: %{x:,}<br>'
                  'Citations: %{customdata[0]:,}<br>'
                  'Cit/paper: %{customdata[1]}<extra></extra>',
))
fig_funders.update_layout(
    title='Top 25 Funders by Funded Paper Count',
    xaxis_title='Papers',
    height=600,
    yaxis=dict(autorange='reversed'),
    margin=dict(l=300),
)
fig_funders.show()

# ── 9b. Funding rate by country ───────────────────────────────────────────────
top_cty = funding_cty_df.nlargest(20, 'papers').sort_values('pct_funded')

fig_fund_cty = make_subplots(
    rows=1, cols=2,
    subplot_titles=['% Papers with Funding Acknowledged',
                    'Funded vs Unfunded Papers'],
)
fig_fund_cty.add_trace(go.Bar(
    x=top_cty['pct_funded'], y=top_cty['country'],
    orientation='h', name='% Funded', marker_color='teal',
    hovertemplate='<b>%{y}</b>: %{x}% funded<extra></extra>',
), row=1, col=1)

top_cty_s = top_cty.sort_values('papers')
fig_fund_cty.add_trace(go.Bar(
    x=top_cty_s['funded_papers'], y=top_cty_s['country'],
    orientation='h', name='Funded', marker_color='teal', opacity=0.8,
), row=1, col=2)
fig_fund_cty.add_trace(go.Bar(
    x=top_cty_s['papers'] - top_cty_s['funded_papers'],
    y=top_cty_s['country'],
    orientation='h', name='No funding record', marker_color='lightgrey',
), row=1, col=2)

fig_fund_cty.update_layout(
    barmode='stack', height=550,
    title_text='Funding Coverage by Country (top 20 by paper count)',
)
fig_fund_cty.update_yaxes(autorange='reversed', row=1, col=1)
fig_fund_cty.show()

# ── 9c. Summary table ─────────────────────────────────────────────────────────
total_funded = funders_df['papers'].sum()
print(f'Papers with at least one funder recorded: {total_funded:,}')
print()
print('Top 15 funders:')
display(top_funders[['funder_name','papers','citations','cit_per_paper']].head(15).reset_index(drop=True))

**Reading these charts:**

**9a — Top funders:**  
Paper count per funder (hover for citation impact).  
Funders with high citations-per-paper relative to their paper count are backing
high-impact research.  A paper with multiple funders is counted once per funder.

**9b — Funding coverage by country:**  
The left panel shows what percentage of a country's papers acknowledge external
funding — this reflects both actual funding rates and publisher metadata deposit
practices.  
The right panel stacks funded vs no-record papers; a large grey bar can mean
genuine self-funding *or* incomplete metadata.  
Countries with high % funded and high citations-per-paper (§8c) are particularly
well-supported research environments.

## 7. Save enriched centrality table

In [13]:
out_path = os.path.join(OUTPUT_DIR, 'author_centrality.csv')
cols = ['author_id', 'name', 'institution', 'country', 'community',
        'papers', 'degree', 'degree_centrality', 'betweenness', 'pagerank', 'clustering']
centrality_df[cols].sort_values('betweenness', ascending=False).to_csv(out_path, index=False)
print(f'Saved enriched centrality table → {out_path}')
centrality_df[cols].head(10)

Saved enriched centrality table → ../output/author_centrality.csv


,author_id,name,institution,country,community,papers,degree,degree_centrality,betweenness,pagerank,clustering
0,https://openalex.org/A5039598820,Neha Khandpur,Universidade de São Paulo,BR,5,96,116,0.084119,0.142175,0.007424,0.009748
1,https://openalex.org/A5042007312,Carlos Augusto Monteiro,Universidade de São Paulo,BR,0,156,139,0.100798,0.129257,0.010008,0.012995
2,https://openalex.org/A5059937532,Renata Bertazzi Levy,Universidade de São Paulo,BR,0,143,165,0.119652,0.107005,0.010824,0.012151
3,https://openalex.org/A5009644208,Camila Aparecida Borges,Universidade de São Paulo,BR,0,15,14,0.010152,0.101152,0.000678,0.017414
4,https://openalex.org/A5071183177,Fernanda Rauber,Universidade de São Paulo,BR,0,93,138,0.100073,0.098458,0.007219,0.015323
5,https://openalex.org/A5064834293,Eurídice Martínez Steele,Universidade de São Paulo,BR,0,133,120,0.087020,0.096528,0.008602,0.009033
6,https://openalex.org/A5083070262,Fernanda Helena Marrocos Leite,Universidade de São Paulo,BR,0,17,38,0.027556,0.094776,0.001636,0.013123
7,https://openalex.org/A5048645214,Daniela Silva Canella,Universidade de São Paulo,BR,0,55,46,0.033358,0.080648,0.003363,0.010542
8,https://openalex.org/A5046488279,Nassib Bezerra Bueno,Universidade Federal de Alagoas,BR,10,29,31,0.022480,0.068789,0.002830,0.023066
9,https://openalex.org/A5100735336,Mengxi Du,Tufts University,US,5,39,62,0.044960,0.062690,0.003297,0.020307


## 9. Export interactive HTML dashboard

Produces a single self-contained HTML file (`output/upf_dashboard.html`) that
can be uploaded to any web server and opened in a browser — no Python or Jupyter
required.  Plotly.js is loaded from CDN; all interactivity (zoom, pan, hover,
"Papers / Citations" toggle buttons) is preserved.

In [ ]:
dashboard_figures = [
    ('Top-N Institutions',                fig_inst),
    ('Top-N Authors',                     fig_auth),
    ('Degree Distribution',               fig_deg),
    ('Co-authorship Network',             fig_network),
    ('Degree vs Betweenness Centrality',  fig_scatter),
    ('Country Collaboration Heatmap',     fig_heatmap),
    ('Author Citation Impact',            fig_cit),
    ('Citations by Community',            fig_comm_cit),
    ('Country Output vs Impact',          fig_cty),
    *([('Top Funders', fig_funders),
       ('Funding Coverage by Country', fig_fund_cty)]
      if not funders_df.empty else []),
]

# ── Build HTML ─────────────────────────────────────────────────────────────────
generated_at = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')

header = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>UPF Bibliometrics — Interactive Dashboard</title>
<style>
  body  {{ font-family: Arial, Helvetica, sans-serif; max-width: 1400px;
          margin: 0 auto; padding: 24px; background: #f4f6f9; color: #333; }}
  h1   {{ font-size: 1.6rem; border-bottom: 3px solid #2c7bb6; padding-bottom: 8px; }}
  h2   {{ font-size: 1.1rem; color: #555; margin: 40px 0 6px; }}
  .chart {{ background: #fff; border-radius: 8px; padding: 12px;
            margin-bottom: 32px; box-shadow: 0 1px 5px rgba(0,0,0,.1); }}
  footer {{ color: #999; font-size: .8rem; margin-top: 40px; border-top: 1px solid #ddd; padding-top: 8px; }}
</style>
</head>
<body>
<h1>UPF Bibliometrics — Interactive Dashboard</h1>
<p>Generated: {generated_at} &nbsp;|&nbsp;
   Source: <a href="https://openalex.org" target="_blank">OpenAlex</a></p>
"""

footer = """
<footer>Generated with <a href="https://plotly.com/python/" target="_blank">Plotly</a>
and the UPF bibliometrics pipeline.</footer>
</body></html>"""

parts = [header]
first = True
for section_title, fig in dashboard_figures:
    chart_html = fig.to_html(
        full_html=False,
        include_plotlyjs='cdn' if first else False,
        config={'displayModeBar': True, 'scrollZoom': False},
    )
    parts.append(f'<h2>{section_title}</h2>\n<div class="chart">\n{chart_html}\n</div>')
    first = False

parts.append(footer)

html_path = os.path.join(OUTPUT_DIR, 'upf_dashboard.html')
with open(html_path, 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(parts))

size_kb = os.path.getsize(html_path) / 1024
print(f'Saved → {html_path}  ({size_kb:.0f} KB)')
print('Upload this single file to any web server to share the dashboard.')